# Análisis de Evaluación del Clasificador — Capítulo 7

Este notebook carga las predicciones generadas por `evaluation/run_evaluation.py` 
y produce las visualizaciones del Capítulo 7 de la tesis:

1. **Heatmap de la matriz de confusión** (seaborn)
2. **Histograma de confianzas por etapa** (deterministic / gemini / fallback)
3. **Curva de calibración** (confianza promedio vs tasa de acierto por bin)

## Prerrequisitos

```bash
# Instalar dependencias
pip install -r evaluation/requirements.txt

# Correr la evaluación primero (requiere corpus real y GEMINI_API_KEY)
python -m evaluation.run_evaluation
```

Si no tenés el corpus real, podés ejecutar contra el fixture sintético:
```bash
python -c "
import asyncio, json, pathlib
from evaluation.corpus import cargar_corpus
from evaluation.run_evaluation import evaluar_corpus, guardar_predicciones
from evaluation.tests.conftest import FakeClassifier, CORPUS_FIXTURE_PATH

async def main():
    corpus = cargar_corpus(CORPUS_FIXTURE_PATH)
    fake = FakeClassifier()
    preds = await evaluar_corpus(corpus, fake)
    guardar_predicciones(preds)

asyncio.run(main())
"
```

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configuración de estilo
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

In [ ]:
# ---------------------------------------------------------------------------
# Carga de predicciones
# ---------------------------------------------------------------------------
PREDICCIONES_PATH = pathlib.Path('evaluation/predicciones.json')

if not PREDICCIONES_PATH.exists():
    raise FileNotFoundError(
        f'No se encontraron predicciones en {PREDICCIONES_PATH}.\n'
        'Ejecutá primero: python -m evaluation.run_evaluation\n'
        'O usá el fixture sintético (ver celda de setup arriba).'
    )

with open(PREDICCIONES_PATH, encoding='utf-8') as f:
    datos = json.load(f)

df = pd.DataFrame(datos)
print(f'Predicciones cargadas: {len(df)} casos')
df.head()

## 1. Heatmap de la Matriz de Confusión

In [ ]:
from evaluation.metrics import CLASES, matriz_confusion

reales = df['categoria_real'].tolist()
predichas = df['categoria_predicha'].tolist()

mc = matriz_confusion(reales, predichas)

# Convertir a array numpy
mc_array = np.array([[mc[r][p] for p in CLASES] for r in CLASES])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    mc_array,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASES,
    yticklabels=CLASES,
    ax=ax,
)
ax.set_xlabel('Categoría Predicha', fontsize=13)
ax.set_ylabel('Categoría Real', fontsize=13)
ax.set_title('Matriz de Confusión del Clasificador Híbrido', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation/figura_matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Histograma de Confianzas por Etapa del Pipeline

In [ ]:
etapas_orden = ['deterministic', 'gemini', 'fallback']
colores = {'deterministic': '#2196F3', 'gemini': '#4CAF50', 'fallback': '#F44336'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, etapa in zip(axes, etapas_orden):
    subset = df[df['etapa'] == etapa]['confianza']
    if len(subset) == 0:
        ax.set_title(f'{etapa.capitalize()}\n(sin casos)')
        continue
    ax.hist(subset, bins=10, range=(0, 1), color=colores[etapa], edgecolor='white', alpha=0.9)
    ax.set_title(f'{etapa.capitalize()}\n({len(subset)} casos)', fontsize=13)
    ax.set_xlabel('Confianza', fontsize=11)
    ax.set_ylabel('Frecuencia', fontsize=11)
    ax.set_xlim(0, 1)
    ax.axvline(0.70, color='red', linestyle='--', linewidth=1.2, label='Umbral revisión (0.70)')
    ax.axvline(0.90, color='orange', linestyle='--', linewidth=1.2, label='Umbral deterministic (0.90)')
    ax.legend(fontsize=9)

fig.suptitle('Distribución de Confianzas por Etapa del Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation/figura_confianzas_por_etapa.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Curva de Calibración (Confianza vs Tasa de Acierto)

In [ ]:
# Bineado de confianzas en 10 bins equiespaciados
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bin_medios = []
tasas_acierto = []
conteos = []

df['acierto'] = df['categoria_real'] == df['categoria_predicha']

for i in range(n_bins):
    mask = (df['confianza'] >= bins[i]) & (df['confianza'] < bins[i + 1])
    if i == n_bins - 1:  # incluir el límite superior en el último bin
        mask = (df['confianza'] >= bins[i]) & (df['confianza'] <= bins[i + 1])
    subset_bin = df[mask]
    if len(subset_bin) > 0:
        bin_medios.append((bins[i] + bins[i + 1]) / 2)
        tasas_acierto.append(subset_bin['acierto'].mean())
        conteos.append(len(subset_bin))

fig, ax = plt.subplots(figsize=(8, 6))

# Línea de calibración perfecta
ax.plot([0, 1], [0, 1], 'k--', label='Calibración perfecta', linewidth=1.5)

# Curva real
scatter = ax.scatter(
    bin_medios, tasas_acierto,
    c=conteos, cmap='viridis', s=100, zorder=5
)
ax.plot(bin_medios, tasas_acierto, 'b-o', alpha=0.7, label='Clasificador híbrido')

plt.colorbar(scatter, ax=ax, label='Cantidad de casos en el bin')
ax.set_xlabel('Confianza promedio del bin', fontsize=13)
ax.set_ylabel('Tasa de acierto', fontsize=13)
ax.set_title('Curva de Calibración del Clasificador Híbrido', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('evaluation/figura_calibracion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nBins de calibración:')
for bm, ta, cnt in zip(bin_medios, tasas_acierto, conteos):
    print(f'  Confianza ≈ {bm:.2f}: tasa_acierto={ta:.3f} ({cnt} casos)')

## 4. Resumen de Métricas Globales

In [ ]:
from evaluation.metrics import (
    exactitud_global, f1_macro, f1_por_clase,
    intervalo_wilson, precision_por_clase, sensibilidad_por_clase
)

exactitud = exactitud_global(reales, predichas)
aciertos = sum(r == p for r, p in zip(reales, predichas))
lower_ic, upper_ic = intervalo_wilson(aciertos, len(reales))
precisiones = precision_por_clase(mc)
sensibilidades = sensibilidad_por_clase(mc)
f1s = f1_por_clase(mc)
f1_m = f1_macro(f1s)

print(f'Exactitud global: {exactitud:.4f} ({exactitud*100:.1f}%)')
print(f'IC Wilson 95%: [{lower_ic:.4f}, {upper_ic:.4f}]')
print(f'F1 Macro: {f1_m:.4f}')
print()
print('Métricas por clase:')
for clase in CLASES:
    print(f'  {clase}:')
    print(f'    Precisión={precisiones[clase]:.4f}, Sensibilidad={sensibilidades[clase]:.4f}, F1={f1s[clase]:.4f}')